#### 汇总md正文内容为一个连续的md文件
1. 找到所有符合模式的md文件，加入md list
2. 遍历md list，处理每一个md
    1. 遍历md每一行，过滤掉非正文信息（在每个md文件的前几行）以及页数分割符，例如：
    # 批次 16: 第 280-304 页
    处理时间: 2026-02-03 19:12:37
    总字符数: 35229
    ---
    {281}------------------------------------------------
    2. 过滤掉图名、表名和图片路径，图表名均为独立一行，以图|附图|附表|表开头，例如：
    图 1-1 浙江宁波保国寺大殿拼柱示意图
    ![图 1-1 浙江宁波保国寺大殿拼柱示意图](./images_all/page_19_Picture_6.png "图 1-1 浙江宁波保国寺大殿拼柱示意图")

    图 1-2 杭州灵隐寺石塔阑额“七朱八白”(五代末)
    ![图 1-2 杭州灵隐寺石塔阑额“七朱八白”(五代末)](./images_all/page_19_Picture_7.png "图 1-2 杭州灵隐寺石塔阑额“七朱八白”(五代末)")

    附表 2 功限比较表
    ![附表 2 功限比较表](./images_all/table_images/-2.png "附表 2 功限比较表")

    3. 保留标题和正文内容，按照顺序加入到一个新的汇总md。标题均以md语法 # XXXX起，且标题为独立一行

3. 汇总的仅包含标题和正文内容的md文件进行格式调整，标题和正文直接空一行，被页码分割符分割的语义不完整的段落，自动拼合，例如：
    《法式》小木作制度中提到许多间广尺寸,《研究》认为它们都以六等材为准,但这是不正

    {285}------------------------------------------------

    确的。如卷七及卷二十一小木作制度与功限《阑槛钩窗》里分别有:"槛面高一尺八寸至二尺。""阑槛一间高一尺八寸,广一丈二尺。
    需要拼接为：    《法式》小木作制度中提到许多间广尺寸,《研究》认为它们都以六等材为准,但这是不正确的。如卷七及卷二十一小木作制度与功限《阑槛钩窗》里分别有:"槛面高一尺八寸至二尺。""阑槛一间高一尺八寸,广一丈二尺。
4. 对汇总文件进行统一半角、圆角符号等格式化操作，用于后续rag chunks切分

In [11]:
import re
import os
from pathlib import Path

class MarkdownMainTextGatherer:
    def __init__(self, md_root_dir, output_file):
        self.md_root_dir = Path(md_root_dir)
        self.output_file = Path(output_file)
        # 匹配模式：图/附图/附表/表 开头的行
        self.fig_table_pattern = re.compile(r'^(图|附图|附表|表)\s?\d+.*')
        # 匹配图片语法：![alt](path)
        self.image_link_pattern = re.compile(r'^!\[.*\]\(.*\)')
        # 匹配页码分隔符：{281}-----------------------
        self.page_divider_pattern = re.compile(r'^\{\d+\}-+')
        # 过滤元数据的关键字
        self.metadata_keywords = ["批次", "处理时间", "总字符数", "---"]

    def find_md_files(self) -> list:
        """查找符合命名模式的md文件并排序"""
        md_files = []
        pattern = re.compile(r'batch\d+_page\d+-\d+_clean\.md$')
        for file_path in self.md_root_dir.rglob('*.md'):
            if pattern.search(file_path.name):
                md_files.append(file_path)
        # 这里的排序很重要，确保文件按顺序拼合
        return sorted(md_files)

    def normalize_text(self, text: str) -> str:
        """格式化操作：统一全角半角符号（根据RAG常规需求，通常保留中文字符习惯）"""
        # 示例：将连续的多个换行缩减为一个（在合并阶段会处理，这里做标点清洗）
        # 如果需要将英文标点转全角或反之，可在此添加映射
        # 这里演示最常见的：去除行首尾多余空格
        return text.strip()

    def is_valid_line(self, line: str) -> bool:
        """判断是否为有效的正文或标题行"""
        line = line.strip()
        if not line:
            return False
        # 过滤元数据行
        if any(keyword in line for keyword in self.metadata_keywords):
            return False
        # 过滤页码分割线
        if self.page_divider_pattern.match(line):
            return False
        # 过滤图表标题
        if self.fig_table_pattern.match(line):
            return False
        # 过滤图片路径
        if self.image_link_pattern.match(line):
            return False
        return True

    def process(self):
        md_files = self.find_md_files()
        print(f"找到 {len(md_files)} 个匹配的 Markdown 文件。")

        all_content_blocks = []

        for file_path in md_files:
            with open(file_path, 'r', encoding='utf-8') as f:
                for line in f:
                    clean_line = line.strip()
                    if self.is_valid_line(clean_line):
                        all_content_blocks.append(clean_line)

        # 执行语义拼合与格式化逻辑
        final_lines = []
        if not all_content_blocks:
            print("没有找到有效内容。")
            return

        for i in range(len(all_content_blocks)):
            current_line = all_content_blocks[i]
            
            # 1. 如果是标题，确保前后格式
            if current_line.startswith('#'):
                # 如果上一行不是空的，可以在逻辑最后处理
                # final_lines.append("\n" + current_line + "\n")
                final_lines.append(current_line)
                continue

            # 2. 语义拼接逻辑：判断当前行是否结束
            # 如果当前行不是以句号、感叹号、问号、引号结尾，且下一行不是标题
            # 则认为下一行是本行的延续
            is_sentence_end = current_line.endswith(('。', '！', '？', '”', '；', '.', '!', '?', '"'))
            
            if not is_sentence_end and (i + 1 < len(all_content_blocks)):
                next_line = all_content_blocks[i+1]
                if not next_line.startswith('#'):
                    # 拼接到当前行，不加换行符
                    all_content_blocks[i+1] = current_line + next_line
                    continue
            
            # 如果是正常结束，或者是最后一行，或者是下一行是标题，则作为独立段落输出
            final_lines.append(current_line)

        # 写入文件
        with open(self.output_file, 'w', encoding='utf-8') as f:
            full_text = "\n\n".join([l for l in final_lines if l.strip()])
            f.write(full_text)
            
        print(f"汇总完成！文件已保存至: {self.output_file}")

# --- Jupyter Cell 运行示例 ---
# 设置你的路径
md_root = r"knowledgeBase\pdfParse\cleaned_data"  # 替换为你的 md 所在文件夹
output_md = r"knowledgeBase\pdfParse\cleaned_data\cleaning\consolidated_corpus.md"

gatherer = MarkdownMainTextGatherer(md_root, output_md)
gatherer.process()

找到 17 个匹配的 Markdown 文件。
汇总完成！文件已保存至: knowledgeBase\pdfParse\cleaned_data\cleaning\consolidated_corpus.md


#### 为章节标题增加路径栈

In [8]:
import json

# --- 参数设置 ---
input_file = r'knowledgeBase\pdfParse\cleaned_data\yingzaofashi_jiedu_v2_toc.jsonl'  # 你的原始文件路径
output_file = r'knowledgeBase\pdfParse\cleaned_data\yingzaofashi_jiedu_v2_toc2.jsonl'  # 输出文件路径

# --- 核心处理逻辑 ---
def process_catalog(in_path, out_path):
    # 用于存储当前的路径层级，Key 是层级(int)，Value 是标题(str)
    # 例如: {1: "第一章", 2: "一、..."}
    current_path_map = {}
    
    processed_data = []

    try:
        with open(in_path, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip():
                    continue
                
                # 1. 解析 JSON
                item = json.loads(line)
                level = item.get("层级")
                title = item.get("章节标题")
                
                # 2. 更新当前层级路径
                # 将当前层级及更深的层级全部清理掉，确保路径正确回溯
                current_path_map[level] = title
                
                # 3. 构建 标题总路径 列表
                # 选取出从 1 到当前 level 的所有标题
                full_path = [current_path_map[i] for i in range(1, level + 1) if i in current_path_map]
                
                # 4. 插入新字段
                item["标题总路径"] = full_path
                processed_data.append(item)

        # 5. 写入新文件
        with open(out_path, 'w', encoding='utf-8') as f:
            for entry in processed_data:
                f.write(json.dumps(entry, ensure_ascii=False) + '\n')
                
        print(f"处理完成！文件已保存至: {out_path}")
        
    except Exception as e:
        print(f"处理过程中出错: {e}")

# 执行函数
process_catalog(input_file, output_file)

处理完成！文件已保存至: knowledgeBase\pdfParse\cleaned_data\yingzaofashi_jiedu_v2_toc2.jsonl


#### 输出images的元数据，包括图索引、图名、路径、所在页码等等

In [8]:
import re
import json
from pathlib import Path
from collections import defaultdict

# ==================== 参数配置区 ====================
# 建议在 Jupyter 中直接修改这些字符串路径
config = {
    "md_root_dir": r"knowledgeBase\pdfParse\cleaned_data",
    "ref_table_path": r"knowledgeBase\pdfParse\cleaned_data\imagesName_Cleaned.md",
    "jsonl_output_path": r"knowledgeBase\pdfParse\cleaned_data\images_metadata.jsonl",
    "update_md_files": False # 是否同步修改原 Markdown 文件内容
}

# ==================== 核心逻辑区 ====================
class MarkdownImageRefiner:
    def __init__(self, cfg):
        self.cfg = {k: (Path(v) if k != "update_md_files" else v) for k, v in cfg.items()}
        self.reference_map = {}  # {归一化索引: {"index": 原索引, "name": 原名称, "page": 页码}}
        self.global_occurrences = defaultdict(list)
        
        self._load_reference_table()

    def _normalize(self, text):
        if not text: return ""
        return re.sub(r'\s+', '', text).replace('：', ':').replace('-', '-')

    def _int_to_chinese(self, n):
        """1-99 整数转中文数字"""
        chars = "零一二三四五六七八九"
        if n < 10: return chars[n]
        if n < 20: return "十" + (chars[n % 10] if n % 10 != 0 else "")
        unit = n % 10
        return chars[n // 10] + "十" + (chars[unit] if unit != 0 else "")

    def _load_reference_table(self):
        """解析参考表，提取 索引、名称、页码"""
        if not self.cfg['ref_table_path'].exists():
            print(f"❌ 找不到参考文件: {self.cfg['ref_table_path']}")
            return

        with open(self.cfg['ref_table_path'], 'r', encoding='utf-8') as f:
            for line in f:
                if '|' not in line or '---' in line or '索引' in line: continue
                # 假设格式: | 索引 | 名称 | 页码 |
                parts = [p.strip() for p in line.split('|') if p.strip()]
                if len(parts) >= 3:
                    idx_part, name_part, page_part = parts[0], parts[1], parts[2]
                    norm_idx = self._normalize(idx_part)
                    self.reference_map[norm_idx] = {
                        "index": idx_part,
                        "name": name_part,
                        "page": page_part
                    }
        print(f"✅ 成功加载参考表，共计 {len(self.reference_map)} 条图名记录。")

    def _extract_index(self, line):
        match = re.search(r'(图|附图|附表|表)\s*([\dA-Za-z\-\.]+)', line)
        return self._normalize(match.group(0)) if match else ""

    def process(self):
        # 1. 扫描所有文件
        md_files = sorted([f for f in self.cfg['md_root_dir'].rglob('*.md') 
                          if re.search(r'batch\d+_page\d+-\d+_clean\.md$', f.name)])
        
        for md_file in md_files:
            with open(md_file, 'r', encoding='utf-8') as f:
                lines = f.readlines()
            
            # 记录该文件中图片和标题的行索引
            img_indices = [i for i, l in enumerate(lines) if re.search(r'!\[.*?\]\((.*?)\)', l)]
            
            for img_idx in img_indices:
                # 在图片行前后2行寻找标题
                search_range = range(max(0, img_idx-2), min(len(lines), img_idx+3))
                for line_idx in search_range:
                    norm_idx = self._extract_index(lines[line_idx])
                    if norm_idx in self.reference_map:
                        img_path_match = re.search(r'!\[.*?\]\((.*?)\)', lines[img_idx])
                        raw_path = img_path_match.group(1).split()[0] if img_path_match else ""
                        
                        # 记录匹配详情
                        self.global_occurrences[norm_idx].append({
                            "file_path": md_file,
                            "img_line_idx": img_idx,
                            "cap_line_idx": line_idx,
                            "raw_img_path": raw_path
                        })
                        break

        # 2. 生成结果 (处理其一、其二逻辑)
        jsonl_data = []
        file_updates = defaultdict(dict) # {file_path: {line_idx: new_content}}

        for norm_idx, occs in self.global_occurrences.items():
            ref = self.reference_map[norm_idx]
            total = len(occs)
            
            for i, occ in enumerate(occs):
                # 确定最终显示名称
                suffix = f"其{self._int_to_chinese(i+1)}" if total > 1 else ""
                final_name = f"{ref['name']}{suffix}"
                
                # 构造 JSONL 数据行
                abs_path = (occ['file_path'].parent / occ['raw_img_path']).resolve()
                jsonl_data.append({
                    "图/表索引": ref['index'],
                    "图/表名称": final_name,
                    "所在页码": int(ref['page']) if str(ref['page']).isdigit() else ref['page'],
                    "本地绝对路径": str(abs_path)
                })
                
                # 如果需要更新原文件内容
                if self.cfg['update_md_files']:
                    standard_block = f"{final_name}\n![{final_name}]({occ['raw_img_path']} \"{final_name}\")\n"
                    file_updates[occ['file_path']][occ['cap_line_idx']] = standard_block
                    file_updates[occ['file_path']][occ['img_line_idx']] = ""

        # 3. 执行写入
        self._write_outputs(jsonl_data, file_updates)

    def _write_outputs(self, jsonl_data, file_updates):
        # 写入 JSONL
        self.cfg['jsonl_output_path'].parent.mkdir(parents=True, exist_ok=True)
        with open(self.cfg['jsonl_output_path'], 'w', encoding='utf-8') as f:
            for item in jsonl_data:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
        print(f"🚀 JSONL 文件已导出: {self.cfg['jsonl_output_path']} (共 {len(jsonl_data)} 条)")

        # 写入原文件修改
        if self.cfg['update_md_files']:
            for f_path, mods in file_updates.items():
                with open(f_path, 'r', encoding='utf-8') as f:
                    lines = f.readlines()
                for ln_idx, content in mods.items():
                    lines[ln_idx] = content
                
                new_content = re.sub(r'\n{3,}', '\n\n', "".join(lines))
                with open(f_path, 'w', encoding='utf-8') as f:
                    f.write(new_content)
            print(f"✨ 原 Markdown 文件已完成标准化重命名。")

# ==================== 执行单元 ====================
refiner = MarkdownImageRefiner(config)
refiner.process()

✅ 成功加载参考表，共计 386 条图名记录。
🚀 JSONL 文件已导出: knowledgeBase\pdfParse\cleaned_data\images_metadata.jsonl (共 428 条)


#### 输出注释索引字典，包括字段，章节、编号、注释内容

In [6]:
import re
import json

# --- 参数设置 ---
input_file = r'knowledgeBase\pdfParse\cleaned_data\allChapters_annotation.md'  # 你的输入文件名
output_file = r'knowledgeBase\pdfParse\cleaned_data\allChapters_annotation.jsonl' # 输出文件名

# --- 工具函数：中文数字转阿拉伯数字 (简单实现) ---
def cn_to_int(cn_str):
    cn_map = {'一': 1, '二': 2, '三': 3, '四': 4, '五': 5, '六': 6, '七': 7, '八': 8, '九': 9, '十': 10}
    if len(cn_str) == 1: return cn_map.get(cn_str, 0)
    if len(cn_str) == 2 and cn_str[0] == '十': return 10 + cn_map.get(cn_str[1], 0)
    if len(cn_str) == 2 and cn_str[1] == '十': return cn_map.get(cn_str[0], 0) * 10
    if len(cn_str) == 3: return cn_map.get(cn_str[0], 0) * 10 + cn_map.get(cn_str[2], 0)
    return 0

# --- 核心逻辑 ---
results = []
stats = {} # 用于存储过程数据
current_chapter_num = 0

# 正则表达式说明：
# 章节：匹配 ## 第(某)章
re_chapter = re.compile(r'^##\s*第([一二三四五六七八九十百]+)章')
# 注释：匹配 - (数字) 或 -(数字) 后的所有内容
re_note = re.compile(r'^\s*-\s*\((\d+)\)\s*(.*)')

with open(input_file, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        
        # 1. 匹配章节
        chapter_match = re_chapter.match(line)
        if chapter_match:
            cn_num = chapter_match.group(1)
            current_chapter_num = cn_to_int(cn_num)
            stats[f"第{cn_num}章"] = 0
            continue
            
        # 2. 匹配注释内容
        note_match = re_note.match(line)
        if note_match and current_chapter_num > 0:
            note_id = int(note_match.group(1))
            content = note_match.group(2).strip()
            
            item = {
                "chapter": current_chapter_num,
                "index": note_id,
                "text": content
            }
            results.append(item)
            
            # 更新统计数据
            chapter_key = list(stats.keys())[-1]
            stats[chapter_key] += 1

# --- 写入 JSONL ---
with open(output_file, 'w', encoding='utf-8') as f:
    for entry in results:
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

# --- 输出过程报告 ---
print("### 处理过程报告 ###")
for ch, count in stats.items():
    print(f"- {ch}: 找到 {count} 条注释")
print("-" * 20)
print(f"总计：生成 {len(results)} 条 JSONL 数据。")

### 处理过程报告 ###
- 第一章: 找到 35 条注释
- 第二章: 找到 31 条注释
- 第三章: 找到 2 条注释
- 第四章: 找到 9 条注释
- 第六章: 找到 5 条注释
- 第八章: 找到 4 条注释
- 第九章: 找到 2 条注释
--------------------
总计：生成 88 条 JSONL 数据。


#### chunks分块与元数据解析类
1. 给定一个仅包含带有层级的章节标题和正文的md文件，经过chunks分块与元数据解析器，输出包含丰富元数据的chunks jsonl文件，并输出中间过程report.md。给定md文件示例、输出jsonl示例如下：
......
# 第三章  铺作

铺作是木构架(大木作)的一部分,因其结构复杂与地位特殊而单立一章。

## 一、概说

当人们走近佛光寺大殿这座唐代殿堂的阶前,走进室内时,其疏朗雄大的斗栱给人以强烈的感受。中国古代建筑远看屋顶、近看斗栱这两个最醒目的特色,在这里得到淋漓尽致的表现。到了编写《营造法式》的年代,斗栱的装饰价值已逐渐被夸张,那些琳琅满目的铺作,有不少部分已失去了原来的结构价值而有了独立的装饰意义:一些房屋开间并不大,也排列了补间铺作;室内纵横罗列的大量斗栱,也是为了承载天花以及烘托皇权和神灵的至高无上。出现这种倾向,当然和北宋时期整个官式建筑追求精巧华丽的总趋势是分不开的,而这种风气沿袭至明清,导致斗栱的累赘程度达到了无以复加的地步。从《法式》所录斗栱图样来看,编者也着眼于复杂的、装饰性强的重栱全计心铺作,对一些简单斗栱和偷心造、单栱造则比较忽视,但恰好是这些简单的做法还较多地保留着斗栱原本的价值和意义。为了使这个曾在我国建筑历史上绽放过异彩的创造不致被追求繁褥豪华的风气所掩盖,我们应该努力让那些简朴真实的铺作恢复其应有的地位。

### （一）出跳承檐

铺作的基本功能是承托悬出的屋檐,其他的承梁、承天花、承平座等功能都是由此衍生的(图 3-1~3-4)。

曾闻老工匠有句口诀叫做"檐不过步",意为出檐不能超过步架,否则就有倾覆的危险, 而在保证出檐安全方面,斗栱起着关键作用。对此,可以用《法式》的相关规定,来对这个原理 进行一番检验:(1)以一座三间小厅堂为例,按《法式》规定可用六等材。如架深用6尺,椽径用0.3尺,则 檐出为3.5尺,飞子出跳为2.1尺,总檐出为5.6尺。如用柱梁作或单斗只替(无斗栱出跳),则 总檐出与架深比为5.6:6等于1:1.07,虽在"檐不过步"范围之内,但比例接近1:1,安全 系数差,如遇大风、大雪、上屋维修甚至地震等突发事件,屋檐垮塌的可能性极大。
......

{”chunk_content“:"当人们走近佛光寺大殿这座唐代殿堂的阶前，走进室内时，其疏朗雄大的斗供给人以强烈的感受......",
"metadata":{
    "chunk_id":"yingzao_fashi_chap3_001", # 务级、可解释 ID
    "chunk_size":389, 
    
    # 一、书籍信息
    "book_info": "《营造法式》解读(修订版)潘谷西 何建中 著",

    # 二、所在章节信息
    "closest_title":"1. 朵数"
    "toc_path": [ "第三章 铺作","一、概述", "（三）补间铺作的布置","1. 朵数"],
    
    #三、多模态
    "has_image":True,
    "images": [
    {
    "figure_id": "图1-1",
    "caption": "浙江宁波保国寺大殿拼柱示意图",
    "path": "本地绝对路径"
    }]，
    #四、正文注解
    "has_annotation":True,
    "annotation":[
    {
    "annotation_id":"chapter_1_index_1"
    "annotation_text":"- (1)《辽宁牛梁河红山文化女神庙与积冢群发掘简报》, 1986年8月《文物》"
    }，
    {
    "annotation_id":"chapter_1_index_2"
    "annotation_text":"- (2) 日僧圆仁:《入唐求法巡礼行记》卷二、卷三。"
    }
    ]
    }
}
2. 包含以下功能：chunks分块器和当前chunks元数据提取器
    1. 子类1：chunks分块类：滑动窗口选择一定行数的正文文本，基于段落分块，不可在同一段落的句子中进行切分，不可出现夸标题的chunk块。切分后的chunksize最大值支持参数设置，默认值为500中文字符。无overlap
    2. 子类2:元数据解析类：对当前切分的chunks进行元数据解析：下面逐字段介绍提取逻辑逻辑
        1. chunk_id，给出自定义前缀参数+章节信息+自动编号，唯一值。记录chunk数量，写入report.md
        1.2 chunk_size：计算字符串长度并写入
        2. book_info：根据给出自定义字符串参数，所有chunks统一写入
        3. closest_title：向上查找，距离当前文本最近的标题项，然后通过匹配标题名称，从外部章节信息jsonl文件的“标题总路径”字段中获取toc_path。记录外部jsonl文件每一行被匹配的次数，写入到report.md中章节索引部分
        4. 子类3:图片解析类：
            1. has_image：编写正则规则，匹配特定模式(图 1-8~1-10)、如图 2-30、(图 9-1、9-2)、（图2-1）、如图三十二、如图所示(图二十八至三十一)、(参见图 2-43)、(附图 2)、(附图4~8)、(表 2-2)、(附表 1)，匹配成功则为true，否则为false
            2. images：如当前文本匹配正则规则，则提取并解析所有的图索引为：图 1-1、图 9-10、附表 3、附图 10格式，加入一个list，注意(图 1-8~1-10)需要解析为1-8、1-9、1-10三张图
            3. 根据图索引list，到给定的所有images数据jsonl文件中，通过匹配“图/表索引”字段，获取“图/表名称”、“本地绝对路径”字段，然后分别填入元数据的images list中。记录外部jsonl文件每一行被匹配的次数，写入到report.md中图片索引部分
        5. 子类4:注解解析类：
            1. has_annotation：编写正则规则，匹配特定模式（<sup>1</sup>），如匹配成功，为true
            2. annotation：通过toc_path的第一个元素，获取到章节信息，然后解析当前（<sup>1</sup>）中的数字作为编号，组成chapter_n_index_m格式作为annotation_id；annotation_text则通过给定的外部引用jsonl文件查找，然后提取写入。记录外部jsonl文件每一行被匹配的次数，写入到report.md中注释索引部分
        6. 下面给出元数据解析器用到的外部jsonl文件示例：
            1. 章节信息jsonl：
            {"章节标题": "第一章  总论", "层级": 1,  "页码": 16, "标题总路径": ["第一章  总论"]}
            {"章节标题": "一、《营造法式》的性质与特点", "层级": 2, "页码": 16, "标题总路径": ["第一章  总论", "一、《营造法式》的性质与特点"]}
            {"章节标题": "（一）“营造法式”是一种建筑工程预算定额", "层级": 3, "标题总路径": ["第一章  总论", "一、《营造法式》的性质与特点", "（一）“营造法式”是一种建筑工程预算定额"], "页码": 16}
            {"章节标题": "（二）李诫《营造法式》的编写体例", "层级": 3, "页码": 17, "标题总路径": ["第一章  总论", "一、《营造法式》的性质与特点", "（二）李诫《营造法式》的编写体例"]}
            2. 注释信息jsonl：
            {"chapter": 1, "index": 1, "text": "《宋会要辑稿》第七十五册,职官三〇。"}
            {"chapter": 1, "index": 2, "text": "《宋会要辑稿》第七十五册,职官三〇。"}
            3. images信息jsonl：
            {"图/表索引": "图 1-1", "图/表名称": "浙江宁波保国寺大殿拼柱示意图", "所在页码": 19, "本地绝对路径": "D:\\postgraduate_study\\graduation_thesis\\llm_rag\\myArAppRag\\knowledgeBase\\pdfParse\\cleaned_data\\images_all\\page_19_Picture_6.png"}
            {"图/表索引": "附表 1", "图/表名称": "南宋永思陵建筑尺度表", "所在页码": 282, "本地绝对路径": "D:\\postgraduate_study\\graduation_thesis\\llm_rag\\myArAppRag\\knowledgeBase\\pdfParse\\cleaned_data\\images_all\\table_images\\-1.png"}


优化逻辑：
1. 图片解析方法：增加“和”、“及”、“与”、“到”等连接词优化
2. 部分正文内容包含(图 2-13-A~2-13-E)，对应的外部jsonl图索引为：图 2-13-A，需要兼容这种情况
3. 注释、图片、章节，与外部jsonl匹配时，去除多余空格、统一符号后，采用计算字符串相似度，相似度>70，即可认为匹配成功
4. 同一chunk内解析到的图片、注解，要去重复，避免因为识别误差添加多次一样的元数据，图片可根据本地绝对路径验重，注释则根据annotation_id
5. 如注释、图片、章节，与外部jsonl匹配失败，report中记录该chunk文本所在md文件的行数，便于人工复核

In [2]:
import json
import re
import os
import unicodedata

# ==========================================
# 1. 通用工具与算法函数
# ==========================================
def cn2arabic(cn_str):
    """中文数字转阿拉伯数字"""
    if str(cn_str).isdigit(): return int(cn_str)
    cn_num_dict = {'零':0, '一':1, '二':2, '三':3, '四':4, '五':5, '六':6, '七':7, '八':8, '九':9, '十':10, '百':100, '千':1000}
    unit_dict = {'十': 10, '百': 100, '千': 1000}
    if cn_str.startswith('十') and len(cn_str) > 1: cn_str = '一' + cn_str
    result, temp = 0, 0
    for char in cn_str:
        if char in unit_dict:
            unit = unit_dict[char]
            if temp == 0: temp = 1
            result += temp * unit
            temp = 0
        elif char in cn_num_dict:
            temp = cn_num_dict[char]
    result += temp
    return result

def normalize_text(text):
    """文本标准化：全半角统一、中英文符号统一、去空格、转小写"""
    if not text: return ""
    text = str(text)
    
    # 1. NFKC 标准化：自动处理全角转半角 (如 １２３ -> 123, ＡＢＣ -> ABC)
    text = unicodedata.normalize('NFKC', text)
    # 2. 转小写
    text = text.lower()
    # 3. 统一中英文标点符号 (将常见的中文标点映射为对应的英文标点)
    trans_table = str.maketrans("，。！？；：‘’“”（）【】", ",.!?;:''\"\"()[]")
    text = text.translate(trans_table)
    # 4. 去除所有空白字符（空格、换行、制表符等）
    text = re.sub(r'\s+', '', text)
    
    return text

def exact_match(query, candidates):
    """基于标准化后的严格字符串比较"""
    if not query: return None
    query_norm = normalize_text(query)
    
    for cand in candidates:
        if query_norm == normalize_text(cand):
            return cand
            
    return None

def expand_image_indices(img_str):
    """展开图号支持连字符和字母"""
    match = re.match(r'([图表]|附[图表])\s*(.+)', img_str.strip())
    if not match: return []
    prefix, content = match.groups()
    
    content = re.sub(r'[，.,和及与]', '、', content)
    content = re.sub(r'[～至到]', '~', content)
    
    if '、' in content:
        return [f"{prefix} {p.strip()}" for p in content.split('、')]
    
    if '~' in content:
        start_str, end_str = [s.strip() for s in content.split('~', 1)]
        if re.search(r'[A-Za-z]$', start_str) and re.search(r'[A-Za-z]$', end_str):
            base = start_str[:-1]
            start_char, end_char = start_str[-1], end_str[-1]
            return [f"{prefix} {base}{chr(i)}" for i in range(ord(start_char), ord(end_char) + 1)]
            
        prefix_main = ""
        if '-' in start_str:
            main_parts = start_str.rsplit('-', 1)
            prefix_main = main_parts[0] + '-'
            start_num_str = main_parts[1]
            end_num_str = end_str.rsplit('-', 1)[-1] if '-' in end_str else end_str
        else:
            start_num_str, end_num_str = start_str, end_str
            
        start_idx, end_idx = cn2arabic(start_num_str), cn2arabic(end_num_str)
        return [f"{prefix} {prefix_main}{i}" for i in range(start_idx, end_idx + 1)]

    num = cn2arabic(content) if not '-' in content and not re.search(r'[A-Za-z]', content) else content
    return [f"{prefix} {num}"]


# ==========================================
# 2. 核心处理类
# ==========================================
class MarkdownProcessor:
    def __init__(self, book_info, prefix_id, max_chunk_size=500):
        self.book_info = book_info
        self.prefix_id = prefix_id
        self.max_chunk_size = max_chunk_size
        
        self.db_chapters = {}    # key: 原始标题
        self.db_annotations = {} # key: (chapter_num, index_num)
        self.db_images = {}      # key: 原始图表索引
        
        self.report_stats = {
            "chunks_total": 0,
            "chapter_hits": {}, "chapter_misses": [],
            "image_hits": {}, "image_misses": [],
            "annotation_hits": {}, "annotation_misses": []
        }

    def load_external_dbs(self, chapters_path, annotations_path, images_path):
        if os.path.exists(chapters_path):
            with open(chapters_path, 'r', encoding='utf-8') as f:
                for line in f:
                    data = json.loads(line)
                    title = data["章节标题"].strip()
                    self.db_chapters[title] = data
                    self.report_stats["chapter_hits"][title] = 0
                    
        if os.path.exists(annotations_path):
            with open(annotations_path, 'r', encoding='utf-8') as f:
                for line in f:
                    data = json.loads(line)
                    anno_key = (data["chapter"], data["index"])
                    self.db_annotations[anno_key] = data
                    self.report_stats["annotation_hits"][anno_key] = 0
                    
        if os.path.exists(images_path):
            with open(images_path, 'r', encoding='utf-8') as f:
                for line in f:
                    data = json.loads(line)
                    idx = data["图/表索引"].strip()
                    self.db_images[idx] = data
                    self.report_stats["image_hits"][idx] = 0
    def _extract_images_from_text(self, text, start_line):
        # 匹配模式保持不变
        pattern = r'[（(]?\s*(?:如图|参见)?(?:所示)?\s*([图表]|附[图表])\s*([A-Za-z一二三四五六七八九十百0-9\-\~\s至及和与到、]+)[）)]?'
        matches = re.finditer(pattern, text)
        
        extracted_images = []
        seen_paths = set()
        
        for match in matches:
            full_raw_text = match.group(0)  # 整个匹配到的文本，如 "(如图 1-1)"
            raw_img_label = match.group(1) + " " + match.group(2) # 拼接出的搜索基准
            
            # 展开图号（如 "图 1~3" 展开为 ["图 1", "图 2", "图 3"]）
            expanded_indices = expand_image_indices(raw_img_label)
            
            for img_idx in expanded_indices:
                matched_key = exact_match(img_idx, self.db_images.keys())
                
                if matched_key:
                    self.report_stats["image_hits"][matched_key] += 1
                    img_data = self.db_images[matched_key]
                    path = img_data.get("本地绝对路径", "")
                    
                    if path not in seen_paths:
                        seen_paths.add(path)
                        extracted_images.append({
                            "figure_id": matched_key,
                            "caption": img_data.get("图/表名称", ""),
                            "path": path
                        })
                else:
                    # 【核心修改】：在警告信息中包含原始捕获文本
                    msg = (f"起始行号 {start_line} | "
                           f"原始文本: 『{full_raw_text}』 | "
                           f"未匹配索引: [{img_idx}]")
                    self.report_stats["image_misses"].append(msg)
                    
        return extracted_images

    def _extract_annotations_from_text(self, text, toc_path, start_line):
        extracted_annotations = []
        seen_ids = set()
        pattern = r'<sup>[（(]?(\d+)[）)]?</sup>'
        matches = re.finditer(pattern, text)
        
        chapter_num = -1
        if toc_path:
            cap_match = re.search(r'第([一二三四五六七八九十百0-9]+)章', toc_path[0])
            if cap_match: chapter_num = cn2arabic(cap_match.group(1))

        for match in matches:
            idx_num = int(match.group(1))
            anno_key = (chapter_num, idx_num)
            anno_id = f"chapter_{chapter_num}_index_{idx_num}"
            
            if anno_id in seen_ids: continue
                
            if anno_key in self.db_annotations:
                self.report_stats["annotation_hits"][anno_key] += 1
                anno_data = self.db_annotations[anno_key]
                seen_ids.add(anno_id)
                extracted_annotations.append({
                    "annotation_id": anno_id,
                    "annotation_text": anno_data.get("text", "")
                })
            else:
                msg = f"起始行号 {start_line} | 未找到注释库映射: 章节 {chapter_num}, 索引 {idx_num}"
                self.report_stats["annotation_misses"].append(msg)
                
        return extracted_annotations

    def process_markdown(self, md_filepath, output_jsonl, report_md):
        with open(md_filepath, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        chunks = []
        current_toc = []
        current_chunk_paragraphs = []
        current_chunk_size = 0
        chunk_start_line = 1
        
        def save_chunk(end_line_num):
            nonlocal current_chunk_paragraphs, current_chunk_size, chunk_start_line
            if not current_chunk_paragraphs: return
            
            chunk_text = "\n".join(current_chunk_paragraphs)
            closest_title = current_toc[-1].lstrip('#').strip() if current_toc else ""
            
            toc_path = []
            matched_title_key = exact_match(closest_title, self.db_chapters.keys()) if closest_title else None
            
            if matched_title_key:
                self.report_stats["chapter_hits"][matched_title_key] += 1
                toc_path = self.db_chapters[matched_title_key].get("标题总路径", [])
            else:
                msg = f"起始行号 {chunk_start_line} | 未找到章节库映射: {closest_title}"
                if closest_title: 
                    self.report_stats["chapter_misses"].append(msg)

            chap_num = "0"
            if toc_path:
                cap_match = re.search(r'第([一二三四五六七八九十百0-9]+)章', toc_path[0])
                if cap_match: chap_num = str(cn2arabic(cap_match.group(1)))

            self.report_stats["chunks_total"] += 1
            chunk_id = f"{self.prefix_id}_chap{chap_num}_{self.report_stats['chunks_total']:03d}"
            
            images = self._extract_images_from_text(chunk_text, chunk_start_line)
            annotations = self._extract_annotations_from_text(chunk_text, toc_path, chunk_start_line)

            chunks.append({
                "chunk_content": chunk_text,
                "metadata": {
                    "chunk_id": chunk_id,
                    "chunk_size": len(chunk_text),
                    "book_info": self.book_info,
                    "closest_title": closest_title,
                    "toc_path": toc_path,
                    "has_image": len(images) > 0,
                    "images": images,
                    "has_annotation": len(annotations) > 0,
                    "annotation": annotations
                }
            })
            
            current_chunk_paragraphs = []
            current_chunk_size = 0

        print("[*] 开始执行 Chunks 分块与元数据解析...")
        for i, line in enumerate(lines, 1):
            line_str = line.strip()
            if not line_str: continue
            
            if not current_chunk_paragraphs:
                chunk_start_line = i
                
            if line_str.startswith('#'):
                save_chunk(i)
                level = len(line_str) - len(line_str.lstrip('#'))
                current_toc = current_toc[:level-1]
                current_toc.append(line_str)
                chunk_start_line = i
                continue
            
            line_len = len(line_str)
            if current_chunk_size + line_len > self.max_chunk_size and current_chunk_size > 0:
                save_chunk(i)
                chunk_start_line = i
            
            current_chunk_paragraphs.append(line_str)
            current_chunk_size += line_len
            
        save_chunk(len(lines))

        print(f"[*] 写入 Chunk 结果到: {output_jsonl}")
        with open(output_jsonl, 'w', encoding='utf-8') as f:
            for c in chunks:
                f.write(json.dumps(c, ensure_ascii=False) + '\n')
                
        self._generate_report(report_md)
        print(f"[*] 处理完成！产出 Chunks 数量: {len(chunks)}。报告已保存至 {report_md}")

    def _generate_report(self, report_md):
        with open(report_md, 'w', encoding='utf-8') as f:
            f.write("# 📚 Chunk分块与元数据解析诊断报告\n\n")
            f.write(f"- 总生成 Chunk 数量: {self.report_stats['chunks_total']}\n\n")
            
            # --- 1. 命中统计 ---
            f.write("## 1. ✅ 外部数据源命中统计 (正文成功关联DB)\n")
            
            f.write("### 📖 章节命中\n")
            for k, v in self.report_stats["chapter_hits"].items():
                if v > 0: f.write(f"- `{k}`: 命中 {v} 次\n")
                
            f.write("\n### 🖼️ 图片命中 (去重后)\n")
            for k, v in self.report_stats["image_hits"].items():
                if v > 0: f.write(f"- `{k}`: 命中 {v} 次\n")
                
            f.write("\n### 📝 注释命中 (去重后)\n")
            for k, v in self.report_stats["annotation_hits"].items():
                if v > 0: f.write(f"- 第{k[0]}章 注释{k[1]}: 命中 {v} 次\n")
                
            # --- 2. 外部数据库未命中 ---
            f.write("\n---\n## 2. 📉 DB中存在但未被正文命中 (需要检查正文是否漏标)\n")
            
            unhit_chapters = [k for k, v in self.report_stats["chapter_hits"].items() if v == 0]
            if unhit_chapters:
                f.write("### 📖 未命中的章节\n")
                for k in unhit_chapters: f.write(f"- {k}\n")
            
            unhit_images = [k for k, v in self.report_stats["image_hits"].items() if v == 0]
            if unhit_images:
                f.write("\n### 🖼️ 未命中的图片\n")
                for k in unhit_images: f.write(f"- {k}\n")
                
            unhit_annos = [k for k, v in self.report_stats["annotation_hits"].items() if v == 0]
            if unhit_annos:
                f.write("\n### 📝 未命中的注释\n")
                for k in unhit_annos: f.write(f"- 第{k[0]}章 注释{k[1]}\n")
            
            if not (unhit_chapters or unhit_images or unhit_annos):
                f.write("🎉 完美！外部数据库提供的所有内容均在正文中被成功引用命中。\n")

            # --- 3. 缺失警告 ---
            f.write("\n---\n## 3. ⚠️ 正文提及但在DB中缺失 (需要检查外部数据库是否遗漏)\n")
            all_misses = (self.report_stats["chapter_misses"] + 
                          self.report_stats["image_misses"] + 
                          self.report_stats["annotation_misses"])
            if not all_misses:
                f.write("🎉 完美！正文中出现的所有元数据标识均在外部数据库中找到了来源。\n")
            else:
                for miss in sorted(set(all_misses)):
                    f.write(f"- {miss}\n")


In [12]:
# ==========================================
CONFIG = {
    # >>> 请按实际情况修改此处路径参数 <<<
    "md_input_path": r"knowledgeBase\pdfParse\cleaned_data\cleaning\consolidated_corpus.md",
    "chapters_jsonl": r"knowledgeBase\pdfParse\cleaned_data\yingzaofashi_jiedu_v2_toc.jsonl",
    "annotations_jsonl": r"knowledgeBase\pdfParse\cleaned_data\allChapters_annotation.jsonl",
    "images_jsonl": r"knowledgeBase\pdfParse\cleaned_data\images_metadata.jsonl",
    
    "output_jsonl": r"knowledgeBase\chunks\yingzaofashi_jieduv2_chunks.jsonl",
    "report_md": r"knowledgeBase\pdfParse\dataAnalysis\chunks_process_report.md",
    
    "book_info_str": "《营造法式》解读(2017年3月修订版) 潘谷西 何建中著",
    "custom_prefix_id": "yingzao_fashi",
    "max_chunk_size": 500
}

# 确保输出目录存在
os.makedirs(os.path.dirname(CONFIG["output_jsonl"]), exist_ok=True)
os.makedirs(os.path.dirname(CONFIG["report_md"]), exist_ok=True)

# 实例化解析管线
processor = MarkdownProcessor(
    book_info=CONFIG["book_info_str"], 
    prefix_id=CONFIG["custom_prefix_id"], 
    max_chunk_size=CONFIG["max_chunk_size"]
)

# 1. 加载外部库建立匹配索引
processor.load_external_dbs(
    CONFIG["chapters_jsonl"], 
    CONFIG["annotations_jsonl"], 
    CONFIG["images_jsonl"]
)

# 2. 执行切分与抽取引擎
if os.path.exists(CONFIG["md_input_path"]):
    processor.process_markdown(
        CONFIG["md_input_path"], 
        CONFIG["output_jsonl"], 
        CONFIG["report_md"]
    )
else:
    print(f"[-] 提示：未找到输入文件 {CONFIG['md_input_path']}，请检查路径。")

[*] 开始执行 Chunks 分块与元数据解析...
[*] 写入 Chunk 结果到: knowledgeBase\chunks\yingzaofashi_jieduv2_chunks.jsonl
[*] 处理完成！产出 Chunks 数量: 501。报告已保存至 knowledgeBase\pdfParse\dataAnalysis\chunks_process_report.md


#### 对术语表提取根据规则提取 稍加1字，并使用开源库添加元数据，形成jsonl少见字术语库，如下：
1. {"汉字":"䫜","UNICODE":"U+4ADC","读音":"ao1","IDS拆字":["幽"，"頁"],"解释":"同“凹”。","替代字":Null,"字形相似汉字":[]}
2. 给定md文件，遍历每一个字符，首先判定是否为汉字，true则进一步判断是否为少见字
    判断少见字方法：加载外部常用字txt文件，如果该字不在txt文件中，则为少见字
        如果为少见字，元数据填充逻辑：1.调用HanziChaizi库获取该字的IDS拆字；2.调取pypinyin库获取拼音；获取unicode码；解释和替代字字段保存Null即可，等人工添加，字形相似汉字，调用from char_similar import std_cal_sim库，和常用字列表进行逐一对比，大于阈值，判定为相似汉字。
3. input：md文件、外部常用字文件、输出路径、外部字库阈值、相似汉字判断阈值
4. output少见字文件，打印jsonl文件长度


In [ ]:
import json
import re
import time
import random
import requests
from bs4 import BeautifulSoup
from hanzi_chaizi import HanziChaizi
from char_similar import std_cal_sim

# 第二个 Cell：核心逻辑类
class IntegratedRareHanziProcessor:
    def __init__(self, config):
        self.config = config
        self.hc = HanziChaizi()
        self.hanzi_re = re.compile(r'[\u4e00-\u9fff\u3400-\u4dbf\U00020000-\U000323af]')
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
            'Referer': 'https://www.hanyuguoxue.com/zidian/'
        }
        self.common_list = []
        self.common_set = set()
        self._load_common_data()

    def _load_common_data(self):
        """加载本地常用字库用于过滤和相似度计算"""
        try:
            with open(self.config["common_chars_path"], 'r', encoding='utf-8') as f:
                lines = [line.strip() for line in f if line.strip()]
                self.common_list = lines[:self.config["common_limit"]]
                self.common_set = set(self.common_list)
            print(f"✅ 成功加载常用字库，规模: {len(self.common_list)}")
        except Exception as e:
            print(f"❌ 加载字库失败: {e}")

    def get_top_k_similar(self, target_char):
        """逻辑保留：计算字形相似度"""
        scores = []
        for cand in self.common_list:
            if cand == target_char: continue
            score = std_cal_sim(target_char, cand, kind="shape", rounded=4)
            scores.append((cand, score))
        scores.sort(key=lambda x: x[1], reverse=True)
        return [item[0] for item in scores[:self.config["top_k"]]]

    def crawl_online_info(self, char):
        """核心爬虫逻辑：获取在线释义和拆字"""
        url = f"https://www.hanyuguoxue.com/zidian/search?words={char}&type=all"
        info = {"读音": "", "UNICODE": "", "拆字": [], "基本解释": "", "来源": url}
        
        try:
            resp = requests.get(url, headers=self.headers, timeout=10)
            resp.encoding = 'utf-8'
            if resp.status_code == 200:
                soup = BeautifulSoup(resp.text, 'html.parser')
                
                # 1. 解析概述（读音、Unicode、拆字）
                summary = soup.find('div', attrs={'data-id': '概述'})
                if summary:
                    text = summary.get_text()
                    py_m = re.search(r'拼音是（(.*?)）', text)
                    if py_m: info["读音"] = py_m.group(1)
                    
                    uni_m = re.search(r'UNICODE是(U\+[0-9A-F]+)', text)
                    if uni_m: info["UNICODE"] = uni_m.group(1)
                    
                    chaizi_tag = summary.find(lambda t: t.name == 'p' and "可拆字为" in t.text)
                    if chaizi_tag:
                        em = chaizi_tag.find('em')
                        if em: info["拆字"] = [c.strip() for c in em.text.replace('、', ',').split(',') if c.strip()]

                # 2. 解析意思
                meaning = soup.find('div', attrs={'data-id': '意思'})
                if meaning:
                    explain_div = meaning.find('div', class_='zi-basic-explain')
                    if explain_div:
                        explains = [p.get_text(strip=True) for p in explain_div.find_all('p', class_='explain')]
                        info["基本解释"] = " ".join(explains).replace('◎', '').strip()
                
                info["来源"] = f"汉语字典 {resp.url}"
        except Exception as e:
            print(f"  ! 网络请求异常 {char}: {e}")
        
        return info

    def run(self):
        # 提取文档中的生僻字
        rare_chars = []
        seen = set()
        with open(self.config["input_md_path"], 'r', encoding='utf-8') as f:
            content = f.read()
            for char in content:
                if char not in seen and self.hanzi_re.match(char) and char not in self.common_set:
                    rare_chars.append(char)
                seen.add(char)
        
        print(f"🔍 发现 {len(rare_chars)} 个待处理生僻字")

        with open(self.config["output_jsonl_path"], 'w', encoding='utf-8') as f_out:
            for i, char in enumerate(rare_chars):
                # 获取在线数据
                online_data = self.crawl_online_info(char)
                # 获取本地相似度数据
                similar_chars = self.get_top_k_similar(char)
                
                # 组装最终结果
                result = {
                    "汉字": char,
                    "UNICODE": online_data["UNICODE"] or f"U+{ord(char):04X}",
                    "读音": online_data["读音"],
                    "拆字": online_data["拆字"],
                    "解释": online_data["基本解释"],
                    "替代字": None,
                    "字形相似汉字": similar_chars,
                    "数据来源": online_data["来源"]
                }
                
                f_out.write(json.dumps(result, ensure_ascii=False) + '\n')
                
                if (i + 1) % 5 == 0:
                    print(f"⏳ 已处理 {i+1}/{len(rare_chars)}...")
                
                time.sleep(random.uniform(0.2, 0.5)) # 保护目标网站

        print(f"🚀 任务完成！结果保存至: {self.config['output_jsonl_path']}")

# 第三个 Cell：运行参数
CONFIG = {
    "common_chars_path": r"knowledgeBase\pdfParse\cleaned_data\characters_count8105.txt",
    "input_md_path": r"knowledgeBase\pdfParse\cleaned_data\yingzaofashi_jiedu_v2_term.md",
    "output_jsonl_path": r"knowledgeBase\pdfParse\cleaned_data\rare_hanzi_integrated.jsonl",
    "common_limit": 3500,
    "top_k": 10
}

processor = IntegratedRareHanziProcessor(CONFIG)
processor.run()

✅ 成功加载常用字库，规模: 3500
🔍 发现 158 个待处理生僻字
⏳ 已处理 5/158...
⏳ 已处理 10/158...
⏳ 已处理 15/158...
⏳ 已处理 20/158...
⏳ 已处理 25/158...
⏳ 已处理 30/158...
⏳ 已处理 35/158...
⏳ 已处理 40/158...
⏳ 已处理 45/158...
⏳ 已处理 50/158...
⏳ 已处理 55/158...
⏳ 已处理 60/158...
⏳ 已处理 65/158...
⏳ 已处理 70/158...
⏳ 已处理 75/158...
⏳ 已处理 80/158...
⏳ 已处理 85/158...
⏳ 已处理 90/158...
⏳ 已处理 95/158...
⏳ 已处理 100/158...
⏳ 已处理 105/158...
⏳ 已处理 110/158...
⏳ 已处理 115/158...
⏳ 已处理 120/158...
⏳ 已处理 125/158...
⏳ 已处理 130/158...
⏳ 已处理 135/158...
⏳ 已处理 140/158...
⏳ 已处理 145/158...
⏳ 已处理 150/158...
⏳ 已处理 155/158...
🚀 任务完成！结果保存至: knowledgeBase\pdfParse\cleaned_data\rare_hanzi_integrated.jsonl


#### 生成包含元数据的专业术语库jsonl
示例：
{"术语":"三栱棱间装","解释":"青绿叠晕棱间装的一种, 在两晕棱间装的身内再重复一层外棱之颜色。,"首卷出处":"第14卷 彩画作制度"，","读音":['sān', 'gǒng', 'léng', 'jiān', 'zhuāng']，“少见字注解”：[]}
任意字段无或者空，统一为none
input：term.jsonl、yingzao toc、rare hanzi文件路径
输入文件示例1 term
{"术语": "丁栿", "解释": "用于山面的纵向梁栿。", "首卷出处": 4}
{"术语": "丁华抹颏栱", "解释": "脊部叉手上角内，蜀柱上横向出耍头之栱。", "首卷出处": 4}
{"术语": "丁头栱", "解释": "只有一卷头的半截栱。", "首卷出处": 4}
示例2营造toc：
{"卷编号": [1, 2], "卷名": "总释", "主要内容": "引经据典地诠释各种建筑物和构件（“名物”）的名称，并说明一些几何形的计算方法，以及当时一些定额的计算方法（“总例”）。", "出处": "梁思成注释《营造法式》_李诫 著；梁思成 注释"}
{"卷编号": [3], "卷名": "壕寨制度、石作制度", "主要内容": "“壕寨”大致相当于今天的土石方工程，如地基、筑墙等；“石作制度”则叙述殿阶基（清代称台基）、踏道（台阶）、柱础、石勾栏（石栏杆）等等的做法和雕饰。", "出处": "梁思成注释《营造法式》_李诫 著；梁思成 注释"}
示例文件3 rare 汉字：
{"汉字": "杚", "UNICODE": "U+675A", "读音": "gū、gài", "拆字": ["木", "乞"], "解释": "把东西弄平。", "替代字": null, "字形相似汉字": ["枪", "栏", "杠", "杜", "栓", "枢", "柜", "松", "枫", "杭"], "数据来源": "汉语字典 https://www.hanyuguoxue.com/zidian/zi-26458"}
{"汉字": "纴", "UNICODE": "U+7EB4", "读音": "rèn", "拆字": ["纟", "壬"], "解释": "①织布帛的丝缕。 ②纺织。", "替代字": null, "字形相似汉字": ["纯", "纸", "纽", "纤", "绝", "组", "红", "经", "绕", "纱"], "数据来源": "汉语字典 https://www.hanyuguoxue.com/zidian/zi-32436"}
{"汉字": "枨", "UNICODE": "U+67A8", "读音": "chéng", "拆字": ["木", "长"], "解释": " ②古代门两旁所竖的长木柱，用以防止车过触门。", "替代字": null, "字形相似汉字": ["析", "松", "板", "杉", "枢", "桃", "梆", "构", "柠", "枉"], "数据来源": "汉语字典 https://www.hanyuguoxue.com/zidian/zi-26536"}

output：term_enhanced.jsonl文件路径，各字段要求：
1. 术语、解释出自term
2. 首卷出处查toc文件获取，匹配卷编号提取并组装
3. 少见字查rare文件获取，拼装术语和解释为，遍历每个字符，如果出现在rare文件中，提取汉字、读音、解释字段，加入少见字注解list
4. 拼音采用以下方法，取出多余list嵌套后获取，示例
from pypinyin import pinyin
pinyin_list = pinyin("三栱棱间装", neutral_tone_with_five=True)
print(pinyin_list)#[['sān'], ['gǒng'], ['léng'], ['jiān'], ['zhuāng']]

In [9]:
import json
import re
from pypinyin import pinyin

def process_terms():
    # 1. 加载 TOC卷 数据 (确保 Key 为字符串，方便匹配)
    toc_map = {}
    try:
        with open(INPUT_FILES["toc"], 'r', encoding='utf-8-sig') as f:
            for line in f:
                line = line.strip()
                if not line: continue
                data = json.loads(line)
                vols = data.get("卷编号", [])
                name = data.get("卷名", "none")
                # 将编号统一转为字符串存储，兼容各种输入
                if isinstance(vols, list):
                    for v in vols: toc_map[str(v)] = name
                else:
                    toc_map[str(vols)] = name
    except Exception as e:
        print(f"读取 TOC 出错: {e}")

    # 2. 加载 Rare Hanzi 数据
    rare_map = {}
    try:
        with open(INPUT_FILES["rare"], 'r', encoding='utf-8-sig') as f:
            for line in f:
                line = line.strip()
                if not line: continue
                data = json.loads(line)
                char = data.get("汉字")
                if char:
                    # 预先拼装好字符串：汉字 读音 解释
                    info = f"{char} {data.get('读音', 'none')} {data.get('解释', 'none')}"
                    rare_map[char] = info
    except Exception as e:
        print(f"读取 Rare Hanzi 出错: {e}")

    # 3. 处理 jiedu Term 数据
    processed_count = 0
    with open(INPUT_FILES["term"], 'r', encoding='utf-8-sig') as f_in, \
         open(OUTPUT_FILE, 'w', encoding='utf-8') as f_out:
        
        for line in f_in:
            line = line.strip()
            if not line: continue
            try:
                item = json.loads(line)
            except:
                continue

            # --- 提取字段 ---
            term_text = item.get("术语") or "none"
            explanation = item.get("解释") or "none"
            
            # 提取数字（处理 "4" 或 "第4卷" 这种输入）
            raw_vol = str(item.get("首卷出处", ""))
            vol_digits = "".join(re.findall(r'\d+', raw_vol))
            
            # --- 逻辑1: 匹配卷名 ---
            vol_name = toc_map.get(vol_digits, "none")
            source_str = f"第{vol_digits}卷 {vol_name}" if vol_digits else "none"

            # --- 逻辑2: 拼音处理 ---
            if term_text != "none":
                pinyin_flat = [p[0] for p in pinyin(term_text, neutral_tone_with_five=True)]
            else:
                pinyin_flat = "none"

            # --- 逻辑3: 少见字注解 (拼装字符串列表) ---
            rare_annotations = []
            # 扫描术语和解释中的每个字
            for char in dict.fromkeys(term_text + explanation):
                if char in rare_map:
                    rare_annotations.append(rare_map[char])
            
            if not rare_annotations:
                rare_annotations = None

            # --- 写入 ---
            output_data = {
                "术语": term_text,
                "解释": explanation,
                "首出处卷": source_str,
                "读音": pinyin_flat,
                "少见字注解": rare_annotations
            }
            f_out.write(json.dumps(output_data, ensure_ascii=False) + '\n')
            processed_count += 1

    print(f"处理完成！成功生成 {processed_count} 条数据至 {OUTPUT_FILE}")

# ================= 配置参数 =================
INPUT_FILES = {
    "term": r"knowledgeBase\pdfParse\cleaned_data\yingzaofashi_jiedu_v2_term.jsonl",
    "toc": r"knowledgeBase\pdfParse\cleaned_data\yingzaofashi_toc.jsonl",
    "rare": r"knowledgeBase\pdfParse\cleaned_data\rare_hanzi_integrated.jsonl"
}
OUTPUT_FILE = r"knowledgeBase\pdfParse\cleaned_data\yingzaofashi_jiedu_v2_term_enhanced.jsonl"
# 运行
process_terms()

处理完成！成功生成 769 条数据至 knowledgeBase\pdfParse\cleaned_data\yingzaofashi_jiedu_v2_term_enhanced.jsonl


#### 为生僻字db增加关联术语

In [17]:
# cell: 为生僻字文件添加涉及术语字段
import json
from typing import List, Set, Dict

def add_term_field(terms_file: str, rare_chars_file: str, output_file: str = None):
    """
    为生僻字 JSONL 文件添加“涉及术语”字段
    :param terms_file: 术语 JSONL 文件路径
    :param rare_chars_file: 生僻字 JSONL 文件路径
    :param output_file: 输出文件路径，若为 None 则覆盖原文件
    """
    # 读取所有术语，存入集合（自动去重）
    terms_set: Set[str] = set()
    with open(terms_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            data = json.loads(line)
            term = data.get('术语')
            if term:
                terms_set.add(term)
    print(f"共加载 {len(terms_set)} 个唯一术语")

    # 读取生僻字数据
    rare_data: List[Dict] = []
    with open(rare_chars_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            data = json.loads(line)
            rare_data.append(data)
    print(f"共加载 {len(rare_data)} 个生僻字记录")

    # 为每个生僻字查找涉及术语
    for char_data in rare_data:
        char = char_data.get('汉字')
        if not char:
            continue
        matched_terms = []
        for term in terms_set:
            if char in term:
                matched_terms.append(term)
        # 无匹配时设为 None（JSON 输出为 null）
        char_data['出现在术语'] = matched_terms if matched_terms else None

    # 写回文件
    out_path = output_file if output_file else rare_chars_file
    with open(out_path, 'w', encoding='utf-8') as f:
        for data in rare_data:
            f.write(json.dumps(data, ensure_ascii=False) + '\n')
    print(f"处理完成，已保存至：{out_path}")

# 使用示例：在 cell 中设置文件路径并调用
terms_file = r"knowledgeBase\pdfParse\cleaned_data\yingzaofashi_jiedu_v2_term.jsonl"
rare_file = r"knowledgeBase\pdfParse\cleaned_data\rare_hanzi_integrated.jsonl"
add_term_field(terms_file, rare_file)

共加载 769 个唯一术语
共加载 127 个生僻字记录
处理完成，已保存至：knowledgeBase\pdfParse\cleaned_data\rare_hanzi_integrated.jsonl


#### 对使用将chunks也向量化，从术语向量中检索相关的术语（rag），作为llm清洗时的参考外部知识
要求大模型返回清洗后的正确文本，并输出一个修正清单
这里考虑下需要进行清洗的文件：图名文件？章节文件？注释文件？原bacth.md文件？抽取为待分块的汇总md文件？

#### chunks 文字清洗
错误类型分析：1.字形相似，导致ocr识别错误。发生在图片像素不够、字笔画太多、或者ocr模型的问题如训练不佳、字库不支持识别少见、缺少少见字训练样本等等。本文采用marker的ocr模型。 少见字一般出现领域专业术语或专业字中，2. 语义错误，当排版不佳时容易出现，如竖向文字。或者由于一些局部文字ocr提取失败，部分文本缺失，导致语义不通顺错误。需要结合上下文进行判断（ocr模型较好的话，此问题较少）
动态加载术语库，首先找到和当前chunk最有关联的几个术语及其解释，作为参考。然后再查找术语库中的包含的生僻字，如何查找？根据关键词（模糊匹配？）？根据语义？向量化？如何向量本身的分词效果就不好呢？

prompt = “

In [94]:
OCR_CLEAN_SYSTEM_PROMPT = """
**角色设定**:
你是一位中国传统木构建筑领域的ocr文本清洗助手，专门负责对《营造法式解读》OCR识别产生的文本进行清洗。
你将收到一份来自《营造法式解读》OCR识别得到的md格式【待清洗文本】，以及与该片段内容相关的【出处】、【术语参考】和【少见字参考】三份参考信息。
你的任务是根据提供的参考内容，对【待清洗文本】进行深度清洗和语义校正。

**清洗任务步骤**：
1.专业术语纠正：首先，如某个术语和给到的【相关术语参考】中的内容不符，修正为正确的术语。如果你判断为专业术语，但没有给出参考，保持原样不修改。
2.少见字纠正：其次，阅读少见字参考信息，学习少见字及其相似汉字，然后如果你判断某个字（尤其是和专业领域内的字）是因为OCR识别导致的字形错误，根据【少见字参考】进行纠正。如果你不确定，保持原样不修改。
3.语义和基础错别字纠正（可选）：最后，判断是否还出现其他的基础语义错误和错别字，结合【出处】背景信息酌情简单进行处理，保证顺畅和准确，但不要过度修改。
4.markdown格式修正：确保清洗后文本的markdown格式正确，如公式、列表。

**清洗原则**：
1. 参考优先：在纠正专业术语和少见字时，优先参考提供的【术语参考】和【少见字参考】，确保清洗结果与这些参考信息保持一致。
2. 最小修改原则：仅在必要时进行修改，避免过度清洗。

**输出要求**：
输出为JSON格式，严格符合schema定义的字段要求。

**示例**：
**输入**：
【待清洗文本】：
"殿阁因有平棋、藻井,屋盖的梁栿槫枋都被遮蔽,所以这些构件的加一可不必讲究,只需草草略施斤斧即可,\"草栿\"\"草架\"之名遂由此而来。屋面荷通过椽、槫而传于草伏、角梁,再分别传于柱头铺作和转角铺作,最后由各朵铺作下的柏斗传之于柱头上。\n在草栿与斗拱之间,还要加一道\"压槽枋\",形成周匝交圈的梁垫,从而使屋盖更稳当地坐落在铺作层上,但在现存宋代实例中并无这类压槽方遗存。为了屋顶梁架稳定,草架槫栿之间还须支撑各种木料,即《法式》卷五所载:\"凡乎棋之上,须随樽栿用枋木及矮柱敦掭随宜支樘固济。\"这是没有规格的自由架设,只求坚固即可。至于屋面斜坡作用于槫所产生的水平推力,则由槫下侧的叉手和托脚予以平衡(如梁头上开\"抱槫口\",即由抱槫口抵消水平力,此时托脚已无多大作用)。"
【出处】："《营造法式》解读(2017年3月修订版) 潘谷西 何建中著 第二章木构架，一、宋代官式建筑木构架的基本类型，（二）殿阁式木构架，3.屋盖层
【术语参考】：
1. “平棋”, “解释”: "一种大格子的天花, 格子中间板上饰有花纹。《法式》原文作“平棊”。 “棊”与“棋”相通,现代汉语作“棋”。
2. "草栿", "解释": "即草架梁。"
3. "压槽枋", "解释": "铺作心上、周转之大枋木。作梁垫,上承草栿。", , "栿 fú 房梁：“二门衡～之上皆刻云龙风虎之状。”

【少见字参考】：
1. "汉字": "栱", "UNICODE": "U+6831", "读音": "gǒng", "拆字": ["木", "共"], "解释": "〔枓～〕见“枓”。", "替代字": null, "字形相似汉字": ["棋", "桂", "林", "模", "柑", "桔", "桦", "核", "横", "枯"]
2. "汉字": "槫", "UNICODE": "U+69EB", "读音": "tuán", "拆字": ["木", "專"], "解释": "檩：“柞，……十年中椽，可杂用，二十岁中屋～。” ", "替代字": null, "字形相似汉字": ["槽", "樱", "樟", "橄", "橱", "椿", "楼", "棱", "椒", "梗"]
3. "汉字": "枋", "UNICODE": "U+678B", "读音": "fāng", "拆字": ["木", "方"], "解释": "①古书上说的一种树，木材可做车。 ②方柱形木材。例如～子（亦指棺材）。", "替代字": null, "字形相似汉字": ["柿", "柠", "构", "杭", "朽", "析", "柄", "梆", "榜", "梢"]

**输出**：
 {
    "original_text":"殿阁因有平棋、藻井,屋盖的梁栿槫枋都被遮蔽,所以这些构件的加一可不必讲究,只需草草略施斤斧即可,\"草栿\"\"草架\"之名遂由此而来。屋面荷通过椽、槫而传于草伏、角梁,再分别传于柱头铺作和转角铺作,最后由各朵铺作下的柏斗传之于柱头上。\n在草栿与斗拱之间,还要加一道\"压槽枋\",形成周匝交圈的梁垫,从而使屋盖更稳当地坐落在铺作层上,但在现存宋代实例中并无这类压槽方遗存。为了屋顶梁架稳定,草架槫栿之间还须支撑各种木料,即《法式》卷五所载:\"凡乎棋之上,须随樽栿用枋木及矮柱敦掭随宜支樘固济。\"这是没有规格的自由架设,只求坚固即可。至于屋面斜坡作用于槫所产生的水平推力,则由槫下侧的叉手和托脚予以平衡(如梁头上开\"抱槫口\",即由抱槫口抵消水平力,此时托脚已无多大作用)。",
    "references_given": {
        "source": "《营造法式》解读(2017年3月修订版) 潘谷西 何建中著 第二章木构架，一、宋代官式建筑木构架的基本类型，（二）殿阁式木构架，3.屋盖层",
        "term":
                1. “平棋”, “解释”: "一种大格子的天花, 格子中间板上饰有花纹。《法式》原文作“平棊”。 “棊”与“棋”相通,现代汉语作“棋”。
                2. "草栿", "解释": "即草架梁。"
                3. "压槽枋", "解释": "铺作心上、周转之大枋木。作梁垫,上承草栿。", , "栿 fú 房梁：“二门衡～之上皆刻云龙风虎之状。”,
        "rare_char": 
                "1. "汉字": "栱", "UNICODE": "U+6831", "读音": "gǒng", "拆字": ["木", "共"], "解释": "〔枓～〕见“枓”。", "替代字": null, "字形相似汉字": ["棋", "桂", "林", "模", "柑", "桔", "桦", "核", "横", "枯"]
                2. "汉字": "槫", "UNICODE": "U+69EB", "读音": "tuán", "拆字": ["木", "專"], "解释": "檩：“柞，……十年中椽，可杂用，二十岁中屋～。” ", "替代字": null, "字形相似汉字": ["槽", "樱", "樟", "橄", "橱", "椿", "楼", "棱", "椒", "梗"]
                3. "汉字": "枋", "UNICODE": "U+678B", "读音": "fāng", "拆字": ["木", "方"], "解释": "①古书上说的一种树，木材可做车。 ②方柱形木材。例如～子（亦指棺材）。", "替代字": null, "字形相似汉字": ["柿", "柠", "构", "杭", "朽", "析", "柄", "梆", "榜", "梢"]"},
    "is_modified": True,
    "cleaned_text": "殿阁因有平棋、藻井,屋盖的梁栿槫枋都被遮蔽,所以这些构件的加工可不必讲究,只需草草略施斤斧即可,\"草栿\"\"草架\"之名遂由此而来。屋面荷载通过椽、槫而传于草栿、角梁,再分别传于柱头铺作和转角铺作,最后由各朵铺作下的栌斗传之于柱头上。\n在草栿与斗栱之间,还要加一道\"压槽枋\",形成周匝交圈的梁垫,从而使屋盖更稳当地坐落在铺作层上,但在现存宋代实例中并无这类压槽枋遗存。为了屋顶梁架稳定,草架槫栿之间还须支撑各种木料,即《法式》卷五所载:\"凡乎棋之上,须随樽栿用枋木及矮柱敦掭随宜枝樘固济。\"这是没有规格的自由架设,只求坚固即可。至于屋面斜坡作用于槫所产生的水平推力,则由槫下侧的叉手和托脚予以平衡(如梁头上开\"抱槫口\",即由抱槫口抵消水平力,此时托脚已无多大作用)。",
}
"""
OCR_CLEAN_USER_PROMPT=f"""
【待清洗文本】：
【出处】：
【术语参考】：
【少见字参考】：
"""

In [ ]:
from openai import OpenAI
import os

OCR_RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "ocr_cleaning_result",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "original_text": {"type": "string", "description": "原始输入的待清洗文本"},
                "references_given": {
                    "type": "object",
                    "properties": {
                        "source": {"type": "string", "description": "文本出处信息"},
                        "term": {"type": "string", "description": "专业术语信息"},
                        "rare_char": {"type": "string", "description": "少见字信息"}
                    },
                    "required": ["source", "term", "rare_char"],
                    "additionalProperties": False
                },
                "is_modified": {"type": "boolean", "description": "是否对原文进行了任何修改"},
                "cleaned_text": {"type": "string", "description": "清洗并修正后的最终文本"}
            },
            "required": ["original_text", "references_given", "is_modified", "cleaned_text"],
            "additionalProperties": False
        }
    }
}

API_KEY=os.getenv("SEU_API_KEY") 
BASE_URL="http://10.128.202.100:3010/v1"
MODEL_NAME="qwen3.5-flash" 

client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
) 
    
completion = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": OCR_CLEAN_SYSTEM_PROMPT},
        {"role": "user", "content": OCR_CLEAN_USER_PROMPT},
    ],
    response_format=OCR_RESPONSE_FORMAT
)

# 获取返回的文本内容
result_text = completion.choices[0].message.content
print(f"原始返回：{result_text}")

2026-03-04 23:09:28,796 - _client.py[line:1025] - INFO: HTTP Request: POST http://10.128.202.100:3010/v1/chat/completions "HTTP/1.1 200 OK"


原始返回：{
  "姓名": "刘五",
  "年龄": 25
}
姓名：未知
年龄：未知
兴趣爱好：未提供


In [ ]:
from openai import OpenAI
import os
import json

API_KEY = os.getenv("SEU_API_KEY")
BASE_URL = "http://10.128.202.100:3010/v1"
MODEL_NAME = "qwen3.5-flash" 

client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
) 
try:
    # 使用标准的 create 方法，而不是 parse
    completion = client.chat.completions.create(  # 注意：这里是 create 不是 parse
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "提取姓名、年龄和兴趣爱好信息，以JSON格式输出。"},
            {"role": "user", "content": "我叫刘五，今年25岁。我爱好打篮球，喜欢吃鸡。"},
        ],
        response_format={"type": "json_object"},  # 使用 json_object
        temperature=0.3
    )
    
    # 获取返回的文本内容
    result_text = completion.choices[0].message.content
    print(f"原始返回：{result_text}")
    
    # 手动解析 JSON
    try:
        result = json.loads(result_text)
        
        # 安全地获取值
        name = result.get('name', '未知')
        age = result.get('age', '未知')
        hobby = result.get('hobby', '未提供')
        
        print(f"姓名：{name}")
        print(f"年龄：{age}")
        print(f"兴趣爱好：{hobby}")
        
    except json.JSONDecodeError as e:
        print(f"JSON 解析失败：{e}")
        print(f"原始内容：{result_text}")

except Exception as e:
    print(f"调用 API 时出错：{e}")
    if hasattr(e, 'response'):
        print(f"响应内容：{e.response.text}")

2026-03-04 23:01:55,721 - _client.py[line:1025] - INFO: HTTP Request: POST http://10.128.202.100:3010/v1/chat/completions "HTTP/1.1 200 OK"


原始返回：{
  "姓名": "刘五",
  "年龄": 25,
  "兴趣爱好": [
    "打篮球",
    "吃鸡"
  ]
}
姓名：未知
年龄：未知
兴趣爱好：未提供


#### 拆字

In [10]:
from hanzi_chaizi import HanziChaizi

hc = HanziChaizi()

char = '關'
parts = hc.query(char)
# 将单引号替换为双引号
result = str(parts).replace("'", '"')
print(result)  # 输出: ["車", "凵", "殳", "土"]

["門", "幺", "幺", "丱"]


#### 拼音

In [14]:
import pypinyin
from pypinyin import pinyin, lazy_pinyin, Style

# 基础拼音转换
text = "𣐕"

# 返回带声调的拼音（默认）
print(pinyin(text))
# 输出: [['zhōng'], ['guó']]

# 返回不带声调的拼音
print(lazy_pinyin(text))
# 输出: ['zhong', 'guo']

[['𣐕']]
['𣐕']
